# Model #

I try to put down a model. I will use the xor-perceptron model as a base and modify that.

first create the lattice, N by N, with Nsol = N**2

I should create a class for the cell object and then the network class is formed by cell objects and when the network is created, each network is associated with a position in the lattice.

Note: for now I'm only considering one possible link between each pair neuron for each direction

In [5]:
import jax
from jax import random as jrd
from jax import numpy as jnp
from jax import debug as jdb
import graph_tool as gt
from graph_tool.all import *
import copy

In [2]:
# parameters of the simulation
par = {'key': jrd.key(1634),    # key for random generation
       'N': 10,
       'int_range': 1,             # interaction range
       'p_self_link': 0.5,
       'length_powerlaw': 1,         # exponent of the wire length power law (DA DEFINIRE!!!)
       'target_set': [0.0,1.0,1.0,0.0],
       'input_set':[[0,0],[0,1],[1,0],[1,1]],
       'N_sol': 10}

In [ ]:
# define a quick function for generating a new random key from par['key'] and update par['key']
def gen_key(par):
    key, subkey = jrd.split(par['key'])     # generate a random key by splitting the key in par
    par['key'] = key                        # update the key in par
    return subkey, par                      # I have to return also par, otherwise, bc of JIT's rules, par['key'] won't update

# quick function to calculate the Eucledian distance between two sites in the lattice
def cdist(a,b,vmap=False):
    if vmap:
        fun = jax.vmap(lambda a,b: jnp.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2),in_axes=0)
        return fun(a,b)
    return jnp.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

class Cell(object):
    
    def __init__(self):     
        self.activation = None     # define the activation function through Kolmogorov-Arnold decomposition or Fourier's     
        self.value = -1    # For now i initialize it to -1, bc I know that can't be 
            
    def generate(self, par):
        self.activation = jax.nn.sigmoid  # DEFINE THIS!!!!
        subkey, par = gen_key(par)                  # generate the subkey for the random generation in the following line
        self.value = jrd.choice(subkey, jnp.arange(2)) # Assign an initial value of either 0 or 1   
    
    def compute(self, input):      # compute the value of the cell given the input and the activation. This will be the value sent to the (eventual) other cells.
        self.value = self.activation(jnp.sum(input))
        

class Network(object):
    
    def __init__(self):             
        self.N = 0                                                  # N - side of the lattice 
        self.lattice = []                                           # lattice
        self.J = None                                               # weight matrix 
        self.C = None                                               # connectivity matrix 
        self.B = None                                               # bias matrix 
        self.D = None                                               # distance matrix - this is computed and stored for faster execution
        self.G = None                                               # Graph
        self.fitness = None                                         # Initialize it to None for avoiding recomputation 
        self.Cchromo = None                                         # Chromosome containing the information about the connectivity of the network (i.e. Cchromo = self.C.reshape(-1))
        
    @property                                                       # I have to make it a property so that it changes dynamically with self.N
    def N_side(self):
        return self.N **2                                           # N_side = N**2 - Number of cells in the lattice (each site in the lattice has one cell in it)
    @property       # Now I define the chromosomes for the genetic algorithm
    def Cchromo(self):     
        if self._Cchromo is None:
            self._Cchromo = jnp.ones((self.N_side**2))
        return self._Cchromo
    @Cchromo.setter
    def Cchromo(self,value):
        self._Cchromo = value
    @property
    def J(self):                                                    
        if self._J is None:
            self._J = jnp.ones((self.N_side, self.N_side))
        return self._J
    @J.setter
    def J(self, value):
        self._J = value
    @property
    def C(self):            # I define C from the Cchromosome, so if I modify the Cchromosome, also C gets modified                                                
        if self._C is None:
            self._C = jnp.reshape(self.Cchromo,(self.N_side,self.N_side))
        return self._C
    @C.setter
    def C(self, value):
        self._C = value
    @property
    def B(self):                                                    
        if self._B is None:
            self._B = jnp.zeros((self.N_side, self.N_side))
        return self._B
    @B.setter
    def B(self, value):
        self._B = value
    @property
    def D(self):                                                    
        if self._D is None:
            self._D = jnp.zeros((self.N_side, self.N_side))
        return self._D
    @D.setter
    def D(self, value):
        self._D = value
    @property
    def G(self):        # Add a graph-tool object as a property for using graph-tool's methods on the network
        if self._G is None:
            self._G = Graph(jnp.column_stack(jnp.nonzero(self.C)))
        return self._G
    @G.setter
    def G(self,value):
        self._G = value


    def generate(self,par):
        self.N = par['N']                                       # First set the parameters of the Network from par
        for s in range(self.N_side):                             # First generate a cell for each lattice site and assign the former to the latter
            cell = Cell()                                       # Initialize a Cell object
            cell.generate(par)                                  # Generate it - (also par['key] gets updated here)
            cell.S = s                                          # Assign a new attribute 'S' which is the site in the 1D lattice string
            cell.coord = (s // self.N, s % self.N)              # Assign a new attribute 'coord' to the cell object and set it to the coordinates of the lattice site
            self.lattice.append(cell)                         # Assign the generated cell to the lattice site
        jdb.print('Lattice ready.')                                     # Then randomly generate weight, bias and connectivity matrices
        
        subkey, par = gen_key(par)                                      # generate the subkey for the random generation in the following line
        self.J = jrd.uniform(subkey,shape=self.J.shape)                 # uniformly populate the weights in the weight matrix J  
        subkey, par = gen_key(par)                                      # generate the subkey for the random generation in the following line
        self.B = jrd.normal(subkey,shape=self.B.shape)                  # extract from a normal distribution centered in 0 with st. dv. = 1 the biases (spero vada bene fatto così)
        self.D = jnp.array([cdist(self.lattice[c].coord,self.lattice[d].coord,vmap=False) 
                            for c in range(self.N_side) 
                            for d in range(self.N_side)]).reshape((self.N_side,self.N_side))
        jdb.print('C before:{c}',c=self.C)
        subkey, par = gen_key(par)
        prob_matrix = jnp.where(jnp.eye(self.N_side, dtype=bool),        # define a matrix for the link probability for C
                    par['p_self_link'], 1 / self.D).reshape(-1)
        self.Cchromo = jnp.where(jrd.bernoulli(subkey, prob_matrix),1,0)    # Generate first the Cchromosome
        self.C = jnp.reshape(self.Cchromo,(self.N_side,self.N_side))        # And from that C   
        jdb.print('C after:{c}',c=self.C)
        jdb.print('Matrices ready.')
        self.compute_fitness(par)                                       # Already compute the fitness of the network
    
    def compute_fitness(self, par,verb:int=0):
        if self.fitness is None:        # I need this check, bc otherwise I risk adding fitness over fitness
            self.fitness = 0.           # This is to avoid type conflict and to make sure that I'm not computing the fitness of a network that already has it
            for i,input in enumerate(par['input_set']):
                output = self.ff(input)
                if verb > 0:
                    jdb.print('Input: {input} -> {output}',input=input,output=output)
                norm_factor = self.G.num_vertices()**2-self.G.num_vertices()                # normalize by N(N-1)
                target_dist = (par['target_set'][i] - output)**2                            # square distance between network output and target (theoretical) output
                volume_cost = (jnp.sum(self.C.reshape(-1))/norm_factor)**2                     # average wiring volume cost (i.e. # of links)
                length_cost = (jnp.sum((self.C.reshape(-1) * self.D.reshape(-1))**par['length_powerlaw'])
                               /norm_factor)**2                                      # average wiring length cost
                path_cost = (jnp.sum(jnp.array([jnp.sum(i) for i in 
                                                jnp.array(shortest_distance(self.G,directed=True))])  
                                    )/norm_factor)**2          # average shortest path length cost
                if verb > 0:
                    jdb.print('Target distance:{d}',d=target_dist)
                    jdb.print('Volume cost:{d}',d=volume_cost)
                    jdb.print('Length cost:{d}',d=length_cost)
                    jdb.print('Path cost:{d}',d=path_cost)
                self.fitness += jnp.exp(target_dist + volume_cost + length_cost + path_cost) # take the exponential of the sum of the costs (weighted if needed)
            self.fitness /= len(par['input_set'])                                           # normalize over the inputs
    
    def ff(self, input:list, verb:int=0):
        # set the value of the two inputs cells through the input value
        self.lattice[0].value = input[0]                        # cell in the upper left corner of the 2D lattice
        self.lattice[(self.N - 1) * self.N-1].value = input[1]    # cell in the lower left corner of the 2D lattice
        # Precompute the contributions for each cell to optimize the loop (this is incredibly faster)
        contributions = jnp.dot(self.C * self.J, jnp.array([cell.value for cell in self.lattice])) + jnp.sum(self.B, axis=1)      # weight x value + bias
        for s in range(self.N_side):
            cell = self.lattice[s]                      # note: here we're treating self links as any other link
            inbound = contributions[s]     # value to give in input to the cell, which then computes its value through the activation
            cell.compute(inbound)           # pass all the contributions through the activation function to determine the value of the cell
        if verb > 0: 
            return [c.value for c in self.lattice]
        output = self.lattice[self.N_side-1].value         # the cell in the right lower corner is the output
        return output
            

Now I define the genetic algorithm for the evolution:

In [ ]:
############# GENETIC ALGORITHM #################
def crossover(par1:Network,par2:Network,par:dict):      # PER ORA IMPLEMENTO SOLO PER CChromo
    # Implements the crossover ricombination given 2 parent Networks
    # and returns 2 offspring Networks.
    if len(par1.Cchromo) != len(par2.Cchromo):
        jdb.print('Error: length mismatch between the chromosomes of the two parents!')
        raise ValueError
    offspring
    subkey, par = gen_key(par)
    cut_idx = jrd.choice(subkey,jnp.arange(len(par1.Cchromo)))  # randomly pick where to cut the chromosomes
    

def evolution(par:dict, verb:int=1,early_stop:bool=True):
    if par(['N_sol']) % 2 != 0:     # N_sol must be even
        jdb.print('Error:The number of solutions N_sol must be an even positive number.')
        raise ValueError
    # Generate the initial batch of solutions
    solutions = []
    for _ in range(par['N_sol']):
        # Generate a solution
        n = Network()
        n.generate(par)     # already computes also the fitness value
        solutions.append(n)
    solutions = jnp.array(solutions)
    # Initiate some container for statistic
    Fmean_values = []
    
    ##  EVOLUTION
    for iter in range(par['n_iter']):
        # Compute the mean fitness for statistics
        mean_fit = jnp.sum(jnp.array([sol.fitness for sol in solutions])) / par['N_sol']    # Here I don't have to divide also by 4, because I've already done it in the compute_fitness method
        # Save the old solution set for possible early stopping
        solutions_old = []
        for sol in solutions:
            solutions_old.append(copy.deepcopy(sol))
        # SELECTION
        solutions = jnp.sort(solutions,key=lambda sol: sol.fitness)    # sort in ascending order based on fitness
        subkey, par = gen_key(par)
        parents_idx = jrd.choice(subkey,jnp.arange(par['N_sol']),                       # must be with replacement, since the same individual 
                                 shape=(par['N_sol']/2,2),replace=True,                 # can be chosen to be parent multiple times
                                 p=jnp.array([1/sol.fitness for sol in sols])           # p: probability of being chosen as a parent, inversly proportional to the fitness
                                 /jnp.sum(jnp.array([sol.fitness for sol in sols]))) 
        
        
        

In [ ]:
# Execution
# Notes: put N as a static argname in .jit()
# REMEMBER TO DEAL WITH THE BOUNDARY CONDITIONS ON THE LATTICE